<a href="https://colab.research.google.com/github/Raghukilli77/25P35A0577_DriveReady_OPC_2026/blob/main/DAY-03/day03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q langchain langchain-groq



In [2]:

import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)


In [16]:
with open('/test.csv', 'r') as f:
    print(f.read())


question,must_include
How much is a plum cake?,500
Do you make eggless cakes?,eggless
Can I get a cake today evening?,24 hours
Do you deliver to Kakinada?,check with the owner
What is the price of 6 cupcakes?,480



In [4]:

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser


In [5]:

SYSTEM = """ROLE: You are the friendly assistant
for Sweet Crumbs Home Bakery.

FACTS:
Plum cake Rs 500.
Chocolate cake Rs 650.
Cupcake Rs 80 each.
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
Never promise delivery time or place.
If something is unknown, say:
I'll check with the owner.
Work out totals carefully.

FORMAT:
Under 50 words.
Friendly English or Telugu matching the customer."""


prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])


In [6]:

chain = prompt | llm | StrOutputParser()


In [7]:

history = []

def ask(question: str) -> str:

    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))

print(ask("What's my name?"))


Namaskāram Ravi! అవును, మా అన్ని కేక్‌లకు ఎగ్‌లెస్ వేరియంట్‌లు ఉన్నాయి – ప్లమ్, చాక్లెట్, కప్‌కేక్‌లు. మీరు ఏది కావాలో చెప్పండి, ధరలు మరియు ఇతర వివరాలు పంపిస్తాను!
Your name is Ravi.


In [18]:
import csv

def run_tests(path="/test.csv"):
    # Read content and strip leading/trailing whitespace/newlines
    with open(path) as f:
        content = f.read().strip()

    # Split into lines so csv.reader/DictReader can process it
    lines = content.splitlines()
    rows = list(csv.DictReader(lines))
    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()
        passed += ok

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "->",
            answer[:70]
        )

    print(
        f"\n{passed}/{len(rows)} passed "
        f"({passed / len(rows):.0%})"
    )

run_tests()

PASS | How much is a plum cake? -> A plum cake costs Rs 500 (eggless version is also available).
FAIL | Do you make eggless cakes? -> Yes! All our cakes – plum, chocolate and cupcakes – are available in e
FAIL | Can I get a cake today evening? -> Sorry, we need at least 24 hours’ notice for any cake order, so today 
PASS | Do you deliver to Kakinada? -> I'll check with the owner and get back to you!
PASS | What is the price of 6 cupcakes? -> Sure! Six cupcakes would cost Rs 480. Let me know if you’d like to pla

3/5 passed (60%)


In [19]:

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)


Yes! All our cakes—including plum, chocolate, and cupcakes—are available in egg‑less versions. Let us know which one you’d like!
